# Übung 00 – Basissicherung: Colab, pandas und ein solider Datencheck

## Wofür diese Übung gedacht ist

Dieses Notebook sichert die Grundhandlungen, die ich für die folgenden Modellierungsübungen brauche: ein Notebook ausführen, Daten aus einer dokumentierten Quelle laden, Spalten verstehen, Datenqualität prüfen und einfache Visualisierungen sinnvoll lesen. Es ist **keine Prüfung**. Wenn etwas hier noch nicht sicher sitzt, ist genau jetzt der richtige Zeitpunkt, um nachzufragen und die Wiederholungsmaterialien zu nutzen.

## Datensatz und Quelle

Wir verwenden die roten Vinho-Verde-Weine aus dem offiziellen UCI Machine Learning Repository.

| Angabe | Quelle |
|---|---|
| Offizielle Dokumentation | https://archive.ics.uci.edu/dataset/186/wine+quality |
| Direkter Download | https://archive.ics.uci.edu/static/public/186/wine+quality.zip |
| DOI | https://doi.org/10.24432/C56S3T |
| Zitierform | Cortez, P., Cerdeira, A., Almeida, F., Matos, T. & Reis, J. (2009). *Wine Quality* [Dataset]. UCI Machine Learning Repository. |
| Lizenz laut UCI | CC BY 4.0 |

Ich nutze den Datensatz hier nur für **Data Understanding und Exploration**. Aus einer Korrelation oder Verteilung leite ich noch keine Ursache und keine Handlungsempfehlung ab.

In [ ]:
# Ich halte alle Imports am Anfang zusammen. Dadurch ist transparent,
# welche Bibliotheken mein Notebook benötigt und ich kann Fehler schneller eingrenzen.
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

# Die feste Zufallszahl macht zufällige Aufteilungen und Modellresultate reproduzierbar.
SEED = 42
np.random.seed(SEED)

# Diese Darstellung ist für die Analyse in Colab gut lesbar.
pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid', context='notebook')

DATA_DIR = Path('daten')
DATA_DIR.mkdir(exist_ok=True)


def download_and_extract_zip(url: str, label: str) -> Path:
    """Lädt ein offizielles ZIP-Archiv nur bei Bedarf herunter und entpackt es.

    Die Funktion ist absichtlich im Notebook sichtbar: Studierende sollen erkennen,
    dass die Datenquelle nicht manuell und nicht über einen lokalen Pfad bereitgestellt wird.
    """
    zip_path = DATA_DIR / f'{label}.zip'
    extract_dir = DATA_DIR / label

    if not zip_path.exists():
        print(f'Lade Daten von: {url}')
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        zip_path.write_bytes(response.content)
    else:
        print(f'Verwende vorhandenes Archiv: {zip_path}')

    if not extract_dir.exists():
        extract_dir.mkdir(parents=True)
        with ZipFile(zip_path) as archive:
            archive.extractall(extract_dir)

    return extract_dir

## 1. Daten reproduzierbar laden

Ich lade das ZIP-Archiv direkt von der offiziellen UCI-Quelle. Der Dateiname im Archiv ist bekannt; bei unbekannten oder wechselnden Quellen würde ich zuerst die Archivstruktur ausgeben, statt blind einen Dateinamen anzunehmen.

In [ ]:
SOURCE_URL = 'https://archive.ics.uci.edu/static/public/186/wine+quality.zip'
extract_dir = download_and_extract_zip(SOURCE_URL, 'wine_quality')

csv_path = extract_dir / 'winequality-red.csv'
assert csv_path.exists(), f'Die erwartete Datei fehlt: {csv_path}'

# Der UCI-Datensatz nutzt Semikolon als Trennzeichen. Ohne sep=';' entstünde nur eine einzige Spalte.
df = pd.read_csv(csv_path, sep=';')
print(f'Datensatzform: {df.shape[0]:,} Zeilen und {df.shape[1]} Spalten')
df.head()

## 2. Erster Datencheck: Was liegt überhaupt vor?

Ich prüfe zuerst Struktur und Datenqualität, bevor ich irgendeine Grafik interpretiere. Besonders wichtig sind Datentypen, fehlende Werte und mögliche Dubletten. Ein Modell kann nur mit den Informationen arbeiten, die tatsächlich in den Daten stehen.

In [ ]:
# info() zeigt Datentypen und nicht-leere Werte. Das ist oft der schnellste erste Plausibilitätscheck.
df.info()

quality_report = pd.DataFrame({
    'datentyp': df.dtypes.astype(str),
    'fehlende_werte': df.isna().sum(),
    'anzahl_einzigartiger_werte': df.nunique()
})
quality_report

In [ ]:
print(f'Exakte Dubletten: {df.duplicated().sum():,}')
print('\\nDeskriptive Kennzahlen:')
display(df.describe().T.round(2))

# Denkfrage: Welche Spalten haben sehr unterschiedliche Größenordnungen?
# Diese Frage wird später wichtig, weil distanzbasierte Verfahren und manche Optimierungsverfahren skalenempfindlich sind.

## 3. Visualisierung 1: Wie verteilt sich die Qualitätsbewertung?

Die Balkengrafik beantwortet eine sehr konkrete Frage: Welche Qualitätswerte kommen häufig und welche selten vor? Seltene Qualitätsstufen wären bei einer späteren Klassifikationsaufgabe ein Hinweis auf Klassenungleichgewicht.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=df, x='quality', color='#3b7ddd', ax=ax)
ax.set(title='Häufigkeit der sensorischen Qualitätswerte', xlabel='Qualitätswert', ylabel='Anzahl Weine')
plt.show()

## 4. Visualisierung 2: Ein Merkmal inhaltlich lesen

Ein Histogramm ist dann sinnvoll, wenn ich die Form einer Verteilung prüfen möchte: typische Werte, Streuung, Schiefe und mögliche Ausreißer. Ich wähle hier `alcohol`, weil die Variable inhaltlich gut interpretierbar ist. Die Grafik beweist **nicht**, dass Alkohol die Qualität verursacht.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data=df, x='alcohol', bins=30, kde=True, color='#1f9d8a', ax=ax)
ax.set(title='Verteilung des Alkoholgehalts', xlabel='Alkohol', ylabel='Anzahl Weine')
plt.show()

## 5. Visualisierung 3: Zusammenhänge als Orientierung, nicht als Kausalitätsnachweis

Eine Korrelations-Heatmap hilft, lineare Zusammenhänge zu entdecken und starke Redundanzen zu erkennen. Sie ersetzt weder fachliche Begründung noch eine spätere Modellvalidierung.

In [ ]:
corr = df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, cmap='vlag', center=0, square=True, ax=ax)
ax.set_title('Lineare Zusammenhänge zwischen numerischen Merkmalen')
plt.show()

## 6. Kurze Transferaufgabe

1. Benenne zwei Spalten, die du vor einer Modellierung besonders prüfen würdest, und begründe dies.
2. Welche Qualitätswerte sind selten? Welche Auswirkung könnte das auf eine Klassifikationsmetrik haben?
3. Warum wäre es methodisch falsch, aus der Heatmap direkt eine kausale Aussage abzuleiten?

> **Merksatz:** Erst Datenstruktur und Datenqualität verstehen, dann modellieren. Jede Grafik braucht eine konkrete Analysefrage.